In [1]:
import pandas as pd
from pandas.tseries.offsets import DateOffset
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import griddata
import datetime as dt
from pathlib import Path
import os
from tqdm import tqdm

In [2]:
# Find nearest index
def find_index(array, x):
    if array.ndim == 1:
        idx = np.argmin(np.abs(array - x))
    elif array.ndim == 2:
        idx = np.unravel_index(np.argmin(np.abs(array - x)), array.shape)
    else:
        raise ValueError("Unsupported array dimensions for find_index function.")
    return idx

In [3]:
#Read in SOA dry dep timeseries files
soa_result_dir = Path('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Timeseries_atm_SOAvars/')
daily_soa_to_open = 'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.*DDF.nc'
nc_daily_soa = xr.open_mfdataset(str(soa_result_dir/daily_soa_to_open),combine='nested')

#Read in DDF for dry dep mass and 
daily_result_dir = Path('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Timeseries_atm_h2')
daily_files_to_open = ['FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.DF_NH4.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.DF_NO3.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.pom_a1DDF.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.pom_a4DDF.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.pom_c1DDF.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.pom_c4DDF.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.dst_a1DDF.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.dst_a2DDF.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.dst_a3DDF.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.dst_c1DDF.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.dst_c2DDF.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.dst_c3DDF.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.dry_deposition_NHx_as_N.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.dry_deposition_NOy_as_N.nc']

# Concatenate directory path with each file name separately
file_paths = [daily_result_dir / file_name for file_name in daily_files_to_open]

# Open multiple netCDF files as a single dataset
nc_daily = xr.open_mfdataset(file_paths, combine='nested')

#merge SOA and other files together
nc_daily_all = xr.merge([nc_daily_soa, nc_daily])
nc_daily_all 

<xarray.Dataset>
Dimensions:                  (time: 7671, lat: 192, lon: 288)
Coordinates:
  * lat                      (lat) float64 -90.0 -89.06 -88.12 ... 89.06 90.0
  * lon                      (lon) float64 0.0 1.25 2.5 ... 356.2 357.5 358.8
  * time                     (time) datetime64[ns] 2002-01-01 ... 2023-01-01
Data variables: (12/34)
    soa1_a1DDF               (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    soa1_a2DDF               (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    soa1_c1DDF               (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    soa1_c2DDF               (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    soa2_a1DDF               (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    soa2_a2DDF               (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    ...                       ...
    dst_a3DDF                (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    dst_c1DDF                (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    dst_c2DDF                (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    dst_c3DDF                (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    dry_deposition_NHx_as_N  (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    dry_deposition_NOy_as_N  (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              FCnudged_f09.mam.Murray.Apr11_01.2002_2023.001
    logname:           demurray
    host:              derecho1
    initial_file:      /glade/campaign/acom/acom-climate/UTLS/shawnh/archive/...
    topography_file:   /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/f...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [29]:
#Add a loop that for each siteId it writes a file to timeseries
output_directory = '/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Daily_Timeseries_NADPsites'

unique_site_ids = nc_daily_nadp_all['siteId'].values

# Initialize tqdm
pbar = tqdm(unique_site_ids, desc="Writing subset files")

# Iterate over each unique siteId
for site_id in pbar:
    # Subset the dataset for the current siteId
    subset_ds = nc_daily_nadp_all.where(nc_daily_nadp_all['siteId'] == site_id, drop=True)
    
    # Construct the filename
    filename = f'{site_id}_DailyTimeseries_DryDep.nc'
    
    # Write the subsetted dataset to the specified directory
    output_path = os.path.join(output_directory, filename)
    subset_ds.to_netcdf(output_path)
    
    # Update tqdm description
    pbar.set_description(f"Writing subset files: {filename}")

Writing subset files: WY96_DailyTimeseries_DryDep.nc: 100%|██████████| 391/391 [4:09:39<00:00, 38.31s/it]  


In [4]:
#Subset by time stamps we have dry dep data for (2017 - 2022)

# Define the time range in the same format as the datetime64 variable
start_date = '2017-01-01T00:00:00.000000000'
end_date = '2023-01-01T23:59:59.999999999'

# Subset the dataset based on time
nc_daily_subset = nc_daily_all.sel(time=slice(start_date, end_date))
nc_daily_subset

<xarray.Dataset>
Dimensions:                  (time: 2192, lat: 192, lon: 288)
Coordinates:
  * lat                      (lat) float64 -90.0 -89.06 -88.12 ... 89.06 90.0
  * lon                      (lon) float64 0.0 1.25 2.5 ... 356.2 357.5 358.8
  * time                     (time) datetime64[ns] 2017-01-01 ... 2023-01-01
Data variables: (12/34)
    soa1_a1DDF               (time, lat, lon) float32 dask.array<chunksize=(2192, 192, 288), meta=np.ndarray>
    soa1_a2DDF               (time, lat, lon) float32 dask.array<chunksize=(2192, 192, 288), meta=np.ndarray>
    soa1_c1DDF               (time, lat, lon) float32 dask.array<chunksize=(2192, 192, 288), meta=np.ndarray>
    soa1_c2DDF               (time, lat, lon) float32 dask.array<chunksize=(2192, 192, 288), meta=np.ndarray>
    soa2_a1DDF               (time, lat, lon) float32 dask.array<chunksize=(2192, 192, 288), meta=np.ndarray>
    soa2_a2DDF               (time, lat, lon) float32 dask.array<chunksize=(2192, 192, 288), meta=np.ndarray>
    ...                       ...
    dst_a3DDF                (time, lat, lon) float32 dask.array<chunksize=(2192, 192, 288), meta=np.ndarray>
    dst_c1DDF                (time, lat, lon) float32 dask.array<chunksize=(2192, 192, 288), meta=np.ndarray>
    dst_c2DDF                (time, lat, lon) float32 dask.array<chunksize=(2192, 192, 288), meta=np.ndarray>
    dst_c3DDF                (time, lat, lon) float32 dask.array<chunksize=(2192, 192, 288), meta=np.ndarray>
    dry_deposition_NHx_as_N  (time, lat, lon) float32 dask.array<chunksize=(2192, 192, 288), meta=np.ndarray>
    dry_deposition_NOy_as_N  (time, lat, lon) float32 dask.array<chunksize=(2192, 192, 288), meta=np.ndarray>
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              FCnudged_f09.mam.Murray.Apr11_01.2002_2023.001
    logname:           demurray
    host:              derecho1
    initial_file:      /glade/campaign/acom/acom-climate/UTLS/shawnh/archive/...
    topography_file:   /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/f...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [16]:
#Determine the sign of each parameter
result_list = []
ds = nc_daily_all

# Iterate over each data variable in the dataset
for var_name in ds.data_vars:
    # Compute the sign of the data variable
    signs = np.sign(ds[var_name].values)
    
    # Determine unique signs
    unique_signs = np.unique(signs)
    
    # Convert unique signs to strings for better representation
    unique_signs_str = ', '.join(['positive' if s > 0 else 'negative' if s < 0 else 'zero' for s in unique_signs])
                                 
    # Append to the result list
    result_list.append({'Variable': var_name, 'Unique_Signs': unique_signs_str})

# Convert the list of dictionaries to a DataFrame
result_df = pd.DataFrame(result_list)

print(result_df)

                   Variable    Unique_Signs
0                soa1_a1DDF        positive
1                soa1_a2DDF  zero, positive
2                soa1_c1DDF  zero, positive
3                soa1_c2DDF  zero, positive
4                soa2_a1DDF        positive
5                soa2_a2DDF  zero, positive
6                soa2_c1DDF  zero, positive
7                soa2_c2DDF  zero, positive
8                soa3_a1DDF        positive
9                soa3_a2DDF  zero, positive
10               soa3_c1DDF  zero, positive
11               soa3_c2DDF  zero, positive
12               soa4_a1DDF        positive
13               soa4_a2DDF  zero, positive
14               soa4_c1DDF  zero, positive
15               soa4_c2DDF  zero, positive
16               soa5_a1DDF        positive
17               soa5_a2DDF  zero, positive
18               soa5_c1DDF  zero, positive
19               soa5_c2DDF  zero, positive
20                   DF_NH4        positive
21                   DF_NO3     

In [8]:
#UNIT CONVERSIONS AND SUMMING
#Dust mass = dst
nc_daily_subset['Dust_mass_sum'] = nc_daily_subset['dst_a1DDF'] + nc_daily_subset['dst_a2DDF'] + nc_daily_subset['dst_a3DDF'] + nc_daily_subset['dst_c1DDF'] + nc_daily_subset['dst_c2DDF'] + nc_daily_subset['dst_c3DDF']

#OC dry = pom, soa (DDF) ** should this include black carbon??
nc_daily_subset['OC_dry_sum'] = (nc_daily_subset['soa1_a1DDF'] + nc_daily_subset['soa1_a2DDF'] + nc_daily_subset['soa1_c1DDF'] 
                             + nc_daily_subset['soa1_c2DDF'] + nc_daily_subset['soa2_a1DDF'] + nc_daily_subset['soa2_a2DDF'] 
                             + nc_daily_subset['soa2_c1DDF'] +  nc_daily_subset['soa2_c2DDF'] + nc_daily_subset['soa3_a1DDF'] 
                             + nc_daily_subset['soa3_a2DDF'] + nc_daily_subset['soa3_c1DDF'] + nc_daily_subset['soa3_c2DDF'] 
                             + nc_daily_subset['soa4_a1DDF'] + nc_daily_subset['soa4_a2DDF'] + nc_daily_subset['soa4_c1DDF'] 
                             + nc_daily_subset['soa4_c2DDF'] + nc_daily_subset['soa5_a1DDF'] + nc_daily_subset['soa5_a2DDF'] 
                             + nc_daily_subset['soa5_c1DDF'] + nc_daily_subset['soa5_c2DDF'] + nc_daily_subset['pom_a1DDF']
                             + nc_daily_subset['pom_a4DDF'] + nc_daily_subset['pom_c1DDF'] + nc_daily_subset['pom_c4DDF'])

# Perform unit conversions
conversions = 86400 * 1000000  # seconds to days and kg to mg per m2
nc_daily_subset = nc_daily_subset.apply(lambda x: x * conversions if x.name in ['Dust_mass_sum', 'OC_dry_sum', 
                                                                                'dry_deposition_NHx_as_N', 'dry_deposition_NOy_as_N',
                                                                               'DF_NH4', 'DF_NO3'] else x)
nc_daily_subset = nc_daily_subset[['Dust_mass_sum', 'OC_dry_sum', 'dry_deposition_NHx_as_N', 'dry_deposition_NOy_as_N', 'DF_NH4', 'DF_NO3']]

In [9]:
#Select cells that correspond to NADP sites: read in NADP lat/long and apply the find nearest function
pathData = '/glade/u/home/demurray/External File Uploads'
os.chdir(pathData)
ntn = pd.read_csv('ntn.csv')

# Only select the cells in .nc that correspond to an NADP site lat/long
subset_list = []
progress_bar = tqdm(total=len(ntn))
for index, row in ntn.iterrows():
    lat = row['latitude']
    lon = 360-(row['longitude']*-1)   # longitude is positive and based on 360 degrees.
    lat_idx = find_index(nc_daily_subset['lat'].values, lat)   # right now we are doing a 'find nearest' calculation, should probably interpolate across grid cell and have exact coordinates represented?
    lon_idx = find_index(nc_daily_subset['lon'].values, lon)
    subset = nc_daily_subset.isel(lat=lat_idx, lon=lon_idx)
    subset['siteId'] = row['siteId']  # Add 'siteId' as a new coordinate/index
    subset = subset.assign_coords(siteId=row['siteId'])
    subset_list.append(subset)
    progress_bar.update(1)
progress_bar.close()

# Concatenate the list of subsets into a new xarray dataset
nc_daily_nadp = xr.concat(subset_list, dim='siteId')
nc_daily_nadp

100%|██████████| 391/391 [00:01<00:00, 229.97it/s]


<xarray.Dataset>
Dimensions:                  (siteId: 391, time: 2192)
Coordinates:
    lat                      (siteId) float64 57.02 56.07 57.02 ... 39.11 40.99
    lon                      (siteId) float64 248.8 248.8 248.8 ... 280.0 253.8
  * time                     (time) datetime64[ns] 2017-01-01 ... 2023-01-01
  * siteId                   (siteId) <U4 'AB32' 'AB34' 'AB36' ... 'WV99' 'WY96'
Data variables:
    Dust_mass_sum            (siteId, time) float64 dask.array<chunksize=(1, 2192), meta=np.ndarray>
    OC_dry_sum               (siteId, time) float64 dask.array<chunksize=(1, 2192), meta=np.ndarray>
    dry_deposition_NHx_as_N  (siteId, time) float64 dask.array<chunksize=(1, 2192), meta=np.ndarray>
    dry_deposition_NOy_as_N  (siteId, time) float64 dask.array<chunksize=(1, 2192), meta=np.ndarray>
    DF_NH4                   (siteId, time) float64 dask.array<chunksize=(1, 2192), meta=np.ndarray>
    DF_NO3                   (siteId, time) float64 dask.array<chunksize=(1, 2192), meta=np.ndarray>

In [11]:
#Turn nc_daily_nadp xarray into a pandas dataframe with similar attributes to the NADP dataset
mod_nadp = nc_daily_nadp.to_dataframe().reset_index()
mod_nadp = mod_nadp.rename(columns = {'time': 'model_time', 'Dust_mass_sum': 'Mod_Dust_mgm2', 'OC_dry_sum': 'Mod_OC_dry_mgm2', 
                                     'dry_deposition_NHx_as_N': 'Mod_NHx_N_dry_mgm2', 'dry_deposition_NOy_as_N' : 'Mod_NOy_N_dry_mgm2',
                                     'DF_NH4' : 'Mod_NH4_dry_mgm2', 'DF_NO3': 'Mod_NO3_dry_mgm2'})

#IMPORTANT STEP: change the time to one day prior (model writes time at end of current day)
mod_nadp['time'] = pd.to_datetime(mod_nadp['model_time'],format='%Y-%m-%d')+DateOffset(days=-1)
mod_nadp.head(10)

,siteId,model_time,Mod_Dust_mgm2,Mod_OC_dry_mgm2,Mod_NHx_N_dry_mgm2,Mod_NOy_N_dry_mgm2,Mod_NH4_dry_mgm2,Mod_NO3_dry_mgm2,lat,lon,time
0,AB32,2017-01-01,7.826760e-05,1.696623,1.426007e+10,8.004925e+09,1.831319e+10,0.0,57.015707,248.75,2016-12-31
1,AB32,2017-01-02,8.404957e-06,0.023710,5.322981e+08,9.876023e+08,6.583737e+08,0.0,57.015707,248.75,2017-01-01
2,AB32,2017-01-03,1.980378e-06,0.008764,4.013177e+08,1.004712e+09,4.946115e+08,0.0,57.015707,248.75,2017-01-02
3,AB32,2017-01-04,9.341370e-05,0.031225,9.051056e+08,4.005296e+09,1.139076e+09,0.0,57.015707,248.75,2017-01-03
4,AB32,2017-01-05,5.994557e-04,0.042066,1.107919e+09,2.072782e+09,1.401818e+09,0.0,57.015707,248.75,2017-01-04
5,AB32,2017-01-06,1.299539e-03,0.055420,1.024902e+09,4.091678e+09,1.240854e+09,0.0,57.015707,248.75,2017-01-05
6,AB32,2017-01-07,1.323915e-04,0.015686,2.266382e+08,4.058706e+09,2.470951e+08,0.0,57.015707,248.75,2017-01-06
7,AB32,2017-01-08,3.101479e-05,0.018303,9.175335e+08,2.279189e+09,1.177826e+09,0.0,57.015707,248.75,2017-01-07
8,AB32,2017-01-09,6.158988e-07,0.009549,8.717684e+08,2.396707e+08,1.098452e+09,0.0,57.015707,248.75,2017-01-08
9,AB32,2017-01-10,8.221446e-06,0.020517,1.353650e+09,1.308588e+09,1.740443e+09,0.0,57.015707,248.75,2017-01-09


In [23]:
#read in timeseries of dry dep and ensure correct/consistent formatting
pathData = '/glade/u/home/demurray/External File Uploads'
os.chdir(pathData)
nadp_df = pd.read_csv('DryDep_clean_final.csv', parse_dates = ['dateOn', 'dateOff'])
nadp_df = nadp_df.sort_values(['siteId', 'dateOn'], ascending = True)

#Convert NH4 and NO3 to -N containing fractions
nadp_df['NO3_N_mgm2'] = nadp_df['NO3_mgm2']*0.368
nadp_df['NH4_N_mgm2'] = nadp_df['NH4_mgm2']*0.87

#Ensure correct date formatting
nadp_df['dateOn'] = pd.to_datetime(nadp_df['dateOn'], format='%m/%d/%Y')
nadp_df['dateOff'] = pd.to_datetime(nadp_df['dateOff'], format='%m/%d/%Y')

#merge with site info and then select relevant columns
nadp_df = pd.merge(nadp_df, ntn, on = 'siteId')
nadp_df = nadp_df[['siteId', 'latitude', 'longitude', 'dateOn', 'dateOff', 'Dry_mgm2', 'Dry_C_mgm2', 'NO3_mgm2', 'NH4_mgm2', 'NO3_N_mgm2', 'NH4_N_mgm2']] 
nadp_df.head(5)

,siteId,latitude,longitude,dateOn,dateOff,Dry_mgm2,Dry_C_mgm2,NO3_mgm2,NH4_mgm2,NO3_N_mgm2,NH4_N_mgm2
0,AZ03,36.0586,-112.184,2017-11-07,2017-12-05,238.84,NaN,NaN,NaN,NaN,NaN
1,AZ03,36.0586,-112.184,2017-12-05,2018-01-02,327.88,NaN,NaN,NaN,NaN,NaN
2,AZ03,36.0586,-112.184,2018-01-02,2018-03-06,173.88,NaN,NaN,NaN,NaN,NaN
3,AZ03,36.0586,-112.184,2018-03-06,2018-04-03,196.00,NaN,NaN,NaN,NaN,NaN
4,AZ03,36.0586,-112.184,2018-04-03,2018-05-01,306.32,50.6,NaN,NaN,NaN,NaN


In [24]:
##Assign sampling intervals to NADP NTN deposition data
nadp_df['SamplingInt'] = pd.Series(dtype='int')
nadp_df['IntTime'] = pd.Series(dtype='int')

sites = nadp_df.siteId.unique()

for i in tqdm(sites, unit = 'sites', total = len(sites), ncols = 100):
    nadp_df.loc[nadp_df.siteId == i,'SamplingInt'] = list(range(0, len(nadp_df.loc[nadp_df.siteId == i]), 1))
    nadp_df.loc[nadp_df.siteId == i, 'IntTime'] = nadp_df.loc[nadp_df.siteId == i, 'dateOff'] - nadp_df.loc[nadp_df.siteId == i, 'dateOn']
nadp_df.head(10)

100%|███████████████████████████████████████████████████████████| 26/26 [00:00<00:00, 374.85sites/s]


,siteId,latitude,longitude,dateOn,dateOff,Dry_mgm2,Dry_C_mgm2,NO3_mgm2,NH4_mgm2,NO3_N_mgm2,NH4_N_mgm2,SamplingInt,IntTime
0,AZ03,36.0586,-112.184,2017-11-07,2017-12-05,238.84,NaN,NaN,NaN,NaN,NaN,0.0,28 days 00:00:00
1,AZ03,36.0586,-112.184,2017-12-05,2018-01-02,327.88,NaN,NaN,NaN,NaN,NaN,1.0,28 days 00:00:00
2,AZ03,36.0586,-112.184,2018-01-02,2018-03-06,173.88,NaN,NaN,NaN,NaN,NaN,2.0,63 days 00:00:00
3,AZ03,36.0586,-112.184,2018-03-06,2018-04-03,196.00,NaN,NaN,NaN,NaN,NaN,3.0,28 days 00:00:00
4,AZ03,36.0586,-112.184,2018-04-03,2018-05-01,306.32,50.6,NaN,NaN,NaN,NaN,4.0,28 days 00:00:00
5,AZ03,36.0586,-112.184,2018-05-01,2018-06-05,1267.70,76.8,NaN,NaN,NaN,NaN,5.0,35 days 00:00:00
6,AZ03,36.0586,-112.184,2018-06-05,2018-08-14,85.40,NaN,NaN,NaN,NaN,NaN,6.0,70 days 00:00:00
7,AZ03,36.0586,-112.184,2018-08-14,2018-09-04,615.93,NaN,NaN,NaN,NaN,NaN,7.0,21 days 00:00:00
8,AZ03,36.0586,-112.184,2018-09-04,2018-10-16,99.12,NaN,NaN,NaN,NaN,NaN,8.0,42 days 00:00:00
9,AZ03,36.0586,-112.184,2018-10-16,2018-11-06,43.47,NaN,NaN,NaN,NaN,NaN,9.0,21 days 00:00:00


In [25]:
#Write loop to assign sampling intervals to  modelled data frame
mod_nadp['SamplingInt'] = pd.Series(dtype='int') # Add a SamplingInt column to the modelled
sites =nadp_df.siteId.unique() 

for i in tqdm(sites, unit = "sites", total = len(sites), ncols = 100):
    sampleInt = nadp_df.loc[nadp_df.siteId==i,'SamplingInt']
    #print(i)
    for j in sampleInt:
       #print(j)
       begDate = pd.Timestamp(nadp_df.loc[(nadp_df.siteId == i) & (nadp_df.SamplingInt == j), 'dateOn'].item())
       endDate = pd.Timestamp(nadp_df.loc[(nadp_df.siteId == i) & (nadp_df.SamplingInt == j), 'dateOff'].item())
       #print(endDate)
       mod_nadp.loc[(mod_nadp.siteId == i) & (mod_nadp.time >= begDate) & (mod_nadp.time < endDate), 'SamplingInt'] = j 

mod_nadp.drop_duplicates(inplace = True)
mod_nadp.dropna(subset = ['SamplingInt'], inplace = True)
mod_nadp.to_csv('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Daily_Timeseries_NADPsites/DryDep_Daily_SamplingIntAssigned.csv')
mod_nadp.head(20)

100%|████████████████████████████████████████████████████████████| 26/26 [00:48<00:00,  1.86s/sites]


,siteId,model_time,Mod_Dust_mgm2,Mod_OC_dry_mgm2,Mod_NHx_N_dry_mgm2,Mod_NOy_N_dry_mgm2,Mod_NH4_dry_mgm2,Mod_NO3_dry_mgm2,lat,lon,time,SamplingInt
30999,AZ03,2017-11-08,0.880946,0.032260,3.657048e+10,1.810084e+10,3.281589e+08,0.0,36.282723,247.5,2017-11-07,0.0
31000,AZ03,2017-11-09,0.090833,0.064230,3.374001e+10,1.668682e+10,5.786662e+08,0.0,36.282723,247.5,2017-11-08,0.0
31001,AZ03,2017-11-10,0.068572,0.308631,4.248820e+10,1.842180e+10,2.320441e+08,0.0,36.282723,247.5,2017-11-09,0.0
31002,AZ03,2017-11-11,0.041739,0.110142,3.732828e+10,2.792251e+10,3.489031e+08,0.0,36.282723,247.5,2017-11-10,0.0
31003,AZ03,2017-11-12,0.194711,0.061425,3.241114e+10,1.794601e+10,3.708523e+08,0.0,36.282723,247.5,2017-11-11,0.0
31004,AZ03,2017-11-13,0.178167,0.319870,5.644590e+10,1.945464e+10,2.745529e+08,0.0,36.282723,247.5,2017-11-12,0.0
31005,AZ03,2017-11-14,0.066475,0.677024,5.495667e+10,2.161553e+10,3.158747e+08,0.0,36.282723,247.5,2017-11-13,0.0
31006,AZ03,2017-11-15,0.027266,0.572828,6.350703e+10,1.891201e+10,2.691084e+08,0.0,36.282723,247.5,2017-11-14,0.0
31007,AZ03,2017-11-16,0.005579,0.791344,6.985787e+10,1.726585e+10,1.777610e+08,0.0,36.282723,247.5,2017-11-15,0.0
31008,AZ03,2017-11-17,0.014172,0.148561,4.153514e+10,1.720116e+10,1.815205e+08,0.0,36.282723,247.5,2017-11-16,0.0


In [27]:
#Then group by siteId and sampling int and sum variables below.
mod_nadp_sum = mod_nadp.groupby(['siteId', 'SamplingInt', 'lat', 'lon'])[['Mod_Dust_mgm2', 'Mod_OC_dry_mgm2', 'Mod_NHx_N_dry_mgm2',	'Mod_NOy_N_dry_mgm2', 'Mod_NH4_dry_mgm2', 'Mod_NO3_dry_mgm2']].sum().reset_index()
mod_nadp_sum

mod_nadp_sum = pd.merge(mod_nadp_sum, nadp_df, how = 'left', on = ['siteId', 'SamplingInt'])
mod_nadp_sum.to_csv('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/SamplingInt_Timeseries_NADPsites/DryDep_Timeseries.SamplingInt_Summed_pairedNADPsites.csv')
mod_nadp_sum.head(20)

,siteId,SamplingInt,lat,lon,Mod_Dust_mgm2,Mod_OC_dry_mgm2,Mod_NHx_N_dry_mgm2,Mod_NOy_N_dry_mgm2,Mod_NH4_dry_mgm2,Mod_NO3_dry_mgm2,...,longitude,dateOn,dateOff,Dry_mgm2,Dry_C_mgm2,NO3_mgm2,NH4_mgm2,NO3_N_mgm2,NH4_N_mgm2,IntTime
0,AZ03,0.0,36.282723,247.5,4.136935,5.307317,1.067390e+12,5.271492e+11,8.388604e+09,0.0,...,-112.184,2017-11-07,2017-12-05,238.84,NaN,NaN,NaN,NaN,NaN,28 days 00:00:00
1,AZ03,1.0,36.282723,247.5,3.240591,1.874730,8.252998e+11,4.915788e+11,9.620683e+09,0.0,...,-112.184,2017-12-05,2018-01-02,327.88,NaN,NaN,NaN,NaN,NaN,28 days 00:00:00
2,AZ03,2.0,36.282723,247.5,38.305914,4.205130,1.806808e+12,1.181571e+12,5.275584e+10,0.0,...,-112.184,2018-01-02,2018-03-06,173.88,NaN,NaN,NaN,NaN,NaN,63 days 00:00:00
3,AZ03,3.0,36.282723,247.5,25.001825,1.157429,9.334537e+11,5.573598e+11,4.111003e+10,0.0,...,-112.184,2018-03-06,2018-04-03,196.00,NaN,NaN,NaN,NaN,NaN,28 days 00:00:00
4,AZ03,4.0,36.282723,247.5,42.062978,3.362554,1.184227e+12,6.620518e+11,5.503055e+10,0.0,...,-112.184,2018-04-03,2018-05-01,306.32,50.6,NaN,NaN,NaN,NaN,28 days 00:00:00
5,AZ03,5.0,36.282723,247.5,69.032225,3.616627,1.430503e+12,1.071736e+12,1.155383e+11,0.0,...,-112.184,2018-05-01,2018-06-05,1267.70,76.8,NaN,NaN,NaN,NaN,35 days 00:00:00
6,AZ03,6.0,36.282723,247.5,62.296485,35.959530,3.490136e+12,2.670049e+12,1.548661e+11,0.0,...,-112.184,2018-06-05,2018-08-14,85.40,NaN,NaN,NaN,NaN,NaN,70 days 00:00:00
7,AZ03,7.0,36.282723,247.5,20.574320,6.563518,8.282944e+11,7.845705e+11,4.171855e+10,0.0,...,-112.184,2018-08-14,2018-09-04,615.93,NaN,NaN,NaN,NaN,NaN,21 days 00:00:00
8,AZ03,8.0,36.282723,247.5,26.907573,7.734508,1.600790e+12,9.828858e+11,5.492982e+10,0.0,...,-112.184,2018-09-04,2018-10-16,99.12,NaN,NaN,NaN,NaN,NaN,42 days 00:00:00
9,AZ03,9.0,36.282723,247.5,4.831179,0.904342,7.307754e+11,3.333637e+11,3.151065e+10,0.0,...,-112.184,2018-10-16,2018-11-06,43.47,NaN,NaN,NaN,NaN,NaN,21 days 00:00:00
